# CSBP441 - LN7: Neural Networks

**Applied Computer Vision | UAE University**

[Open this notebook in Google Colab](https://colab.research.google.com/github/MoyoG/CSBP441/blob/main/docs/notebooks/LN7_Neural_Networks.ipynb)

This notebook accompanies **LN7: Neural Networks**. It starts with one artificial
neuron, builds AND and XOR classifiers, and finishes with a trainable 10-class
handwritten-digit classifier.

## Learning outcomes

By the end of the notebook, you should be able to:

1. Compute a neuron's weighted sum, bias, and activation by hand.
2. Explain the roles of sigmoid, tanh, ReLU, leaky ReLU, and softmax.
3. Count the trainable parameters in fully connected networks.
4. Explain why one perceptron solves AND but cannot solve XOR.
5. Calculate softmax probabilities and cross-entropy loss.
6. Separate training, validation, and test data correctly.
7. Train and evaluate a one-layer image classifier.
8. Interpret accuracy, precision, recall, F1, and a confusion matrix.
9. Use error analysis to propose a specific model or data improvement.


## 1. Setup

The examples use NumPy, Matplotlib, and scikit-learn. In Colab, the notebook
first tries to load MNIST. If that download is unavailable, it uses
scikit-learn's smaller handwritten-digits dataset and resizes each image to
`28 x 28` so the same network code still runs.


In [ ]:
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split, StratifiedKFold

SEED = 441
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["axes.grid"] = False

print("NumPy version:", np.__version__)
print("Random seed:", SEED)


## 2. From image evidence to a trainable decision

Traditional computer vision often uses expert-designed features followed by a
classifier. A neural network learns useful intermediate representations from
data and a task loss.

### Spectrum of supervision

| Setting | Labels available during training | Example |
|---|---|---|
| Unsupervised | No human class labels | Cluster images by visual similarity |
| Weakly supervised | Coarse, incomplete, or noisy labels | An image-level label for an object-localization task |
| Semi-supervised | A small labeled set and a larger unlabeled set | 1,000 labeled images plus 20,000 unlabeled images |
| Fully supervised | A target label for every training example | A digit label for every MNIST image |

The definitions depend on the task. An image-level class label is fully
supervised for classification but weak supervision for pixel segmentation.


## 3. One artificial neuron

For inputs $x_i$, weights $w_i$, and bias $b$, a neuron first calculates

$$
z = \mathbf{x}^T\mathbf{w}+b = \sum_i x_iw_i+b.
$$

An activation function converts $z$ into the output $a=f(z)$.

### Worked example

$$
z=(0.2)(-1.0)+(0.8)(0.6)+(0.5)(0.4)+0.1=0.58.
$$

The sigmoid activation is

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$


In [ ]:
x = np.array([0.2, 0.8, 0.5])
w = np.array([-1.0, 0.6, 0.4])
b = 0.1

products = x * w
z = x @ w + b
sigmoid_output = 1 / (1 + np.exp(-z))

for i, (xi, wi, product) in enumerate(zip(x, w, products), start=1):
    print(f"x{i} * w{i} = {xi:.1f} * {wi:.1f} = {product:+.2f}")
print(f"bias      = {b:+.2f}")
print(f"z         = {z:.2f}")
print(f"sigmoid(z)= {sigmoid_output:.4f}")


## 4. Activation functions

A network needs nonlinear hidden activations. Without them, several affine
layers collapse into one affine transformation:

$$
\mathbf{z}_2=W_2(W_1\mathbf{x}+\mathbf{b}_1)+\mathbf{b}_2
=(W_2W_1)\mathbf{x}+(W_2\mathbf{b}_1+\mathbf{b}_2).
$$

Common choices:

| Activation | Equation | Typical use |
|---|---|---|
| Linear | $f(z)=z$ | Regression output |
| Step | $f(z)=1$ if $z\geq0$, otherwise $0$ | Classical perceptron |
| Sigmoid | $f(z)=1/(1+e^{-z})$ | Binary output probability |
| Tanh | $f(z)=\tanh(z)$ | Zero-centered hidden activation |
| ReLU | $f(z)=\max(0,z)$ | Common hidden activation |
| Leaky ReLU | $f(z)=\max(\alpha z,z)$ | Keeps a small negative-side gradient |
| Softmax | $p_i=e^{z_i}/\sum_j e^{z_j}$ | Multiclass output probabilities |


In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    return np.maximum(0.0, z)


def leaky_relu(z, alpha=0.1):
    return np.where(z >= 0, z, alpha * z)


z_axis = np.linspace(-6, 6, 600)
activations = {
    "Linear": z_axis,
    "Step": (z_axis >= 0).astype(float),
    "Sigmoid": sigmoid(z_axis),
    "Tanh": np.tanh(z_axis),
    "ReLU": relu(z_axis),
    "Leaky ReLU": leaky_relu(z_axis),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (name, values) in zip(axes.ravel(), activations.items()):
    ax.plot(z_axis, values, linewidth=2)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_title(name)
    ax.set_xlabel("z")
    ax.set_ylabel("f(z)")
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Their derivatives explain how gradient-based learning behaves.
sigmoid_values = sigmoid(z_axis)
derivatives = {
    "Linear derivative": np.ones_like(z_axis),
    "Sigmoid derivative": sigmoid_values * (1 - sigmoid_values),
    "Tanh derivative": 1 - np.tanh(z_axis) ** 2,
    "ReLU derivative": (z_axis > 0).astype(float),
    "Leaky ReLU derivative": np.where(z_axis > 0, 1.0, 0.1),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharex=True)
for ax, (name, values) in zip(axes, derivatives.items()):
    ax.plot(z_axis, values, linewidth=2)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("z")
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("A linear activation has derivative 1 everywhere.")
print("Its gradient does not depend on the input, and stacked linear layers stay linear.")


## 5. Counting trainable parameters

For a fully connected layer with $n_{in}$ inputs and $n_{out}$ neurons:

$$
\text{parameters}=n_{in}n_{out}+n_{out}.
$$

The first term counts weights. The second counts one bias per output neuron.


In [ ]:
def dense_parameter_count(layer_sizes):
    rows = []
    total = 0
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        weights = n_in * n_out
        biases = n_out
        total += weights + biases
        rows.append((n_in, n_out, weights, biases, weights + biases))
    return rows, total


architectures = {
    "Two inputs -> one output": [2, 1],
    "4 -> 4 -> 2": [4, 4, 2],
    "3 -> 4 -> 4 -> 2": [3, 4, 4, 2],
    "MNIST linear 784 -> 10": [784, 10],
    "MNIST MLP 784 -> 128 -> 10": [784, 128, 10],
}

for name, sizes in architectures.items():
    rows, total = dense_parameter_count(sizes)
    details = " + ".join(f"{a}x{b}+{b}" for a, b, *_ in rows)
    print(f"{name:32s}: {details} = {total:,}")


## 6. Perceptron: AND works, XOR fails

A perceptron is a binary linear classifier:

$$
\hat y=\operatorname{step}(\mathbf{x}^T\mathbf{w}+b).
$$

Its decision boundary is a single line. AND is linearly separable. XOR is not,
because no single line can separate its two positive points from its two
negative points.


In [ ]:
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1])
y_xor = np.array([0, 1, 1, 0])


def train_perceptron(X, y, epochs=20, learning_rate=0.2):
    weights = np.zeros(X.shape[1])
    bias = 0.0
    mistakes = []
    for _ in range(epochs):
        epoch_mistakes = 0
        for xi, target in zip(X, y):
            prediction = int(xi @ weights + bias >= 0)
            error = target - prediction
            weights += learning_rate * error * xi
            bias += learning_rate * error
            epoch_mistakes += int(error != 0)
        mistakes.append(epoch_mistakes)
    return weights, bias, np.array(mistakes)


def perceptron_predict(X, weights, bias):
    return (X @ weights + bias >= 0).astype(int)


w_and, b_and, mistakes_and = train_perceptron(X_logic, y_and)
w_xor, b_xor, mistakes_xor = train_perceptron(X_logic, y_xor)

print("AND weights and bias:", w_and, b_and)
print("AND predictions:      ", perceptron_predict(X_logic, w_and, b_and))
print("XOR predictions:      ", perceptron_predict(X_logic, w_xor, b_xor))
print("Final XOR mistakes:   ", mistakes_xor[-1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mistakes_and, marker="o", label="AND")
axes[0].plot(mistakes_xor, marker="s", label="XOR")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mistakes")
axes[0].set_title("Perceptron training")
axes[0].legend()
axes[0].grid(alpha=0.25)

for target, marker, color in [(0, "o", "tab:blue"), (1, "s", "tab:red")]:
    mask = y_xor == target
    axes[1].scatter(X_logic[mask, 0], X_logic[mask, 1], s=120,
                    marker=marker, color=color, label=f"XOR class {target}")
axes[1].set_xlim(-0.3, 1.3)
axes[1].set_ylim(-0.3, 1.3)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")
axes[1].set_title("XOR is not linearly separable")
axes[1].legend()
plt.tight_layout()
plt.show()


## 7. A small nonlinear network solves XOR

We use a network with two inputs, four hidden tanh units, and one sigmoid
output:

$$
[x_1,x_2]\rightarrow \text{Dense}(4)\rightarrow\tanh
\rightarrow\text{Dense}(1)\rightarrow\sigma.
$$

The hidden nonlinearities allow the network to combine several linear
boundaries. The code below implements forward propagation and backpropagation
directly with NumPy.


In [ ]:
rng = np.random.default_rng(SEED)
W1 = rng.normal(0, 0.8, size=(2, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0, 0.8, size=(4, 1))
b2 = np.zeros((1, 1))

X = X_logic
y = y_xor.reshape(-1, 1)
learning_rate = 0.15
loss_history = []

for epoch in range(5000):
    # Forward pass
    z1 = X @ W1 + b1
    h = np.tanh(z1)
    z2 = h @ W2 + b2
    y_hat = sigmoid(z2)

    eps = 1e-12
    loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
    loss_history.append(loss)

    # Backpropagation for sigmoid + binary cross-entropy
    dz2 = (y_hat - y) / len(X)
    dW2 = h.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)
    dh = dz2 @ W2.T
    dz1 = dh * (1 - h**2)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

final_probabilities = sigmoid(np.tanh(X @ W1 + b1) @ W2 + b2).ravel()
final_predictions = (final_probabilities >= 0.5).astype(int)

print("XOR probabilities:", np.round(final_probabilities, 4))
print("XOR predictions:  ", final_predictions)
print("XOR ground truth: ", y_xor)

plt.figure(figsize=(7, 3.5))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("XOR MLP training loss")
plt.grid(alpha=0.25)
plt.show()


## 8. Softmax for multiclass classification

For class scores (logits) $z_0,\ldots,z_{C-1}$, softmax calculates

$$
p_i=\frac{e^{z_i}}{\sum_{j=0}^{C-1}e^{z_j}}.
$$

For numerical stability, subtract the largest score before exponentiation:

$$
p_i=\frac{e^{z_i-z_{max}}}{\sum_j e^{z_j-z_{max}}}.
$$

Every probability lies between zero and one, and the probabilities sum to one.


In [ ]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=-1, keepdims=True)


scores = np.array([0.58, 1.20, -0.30])
probabilities = softmax(scores)

for class_index, (score, probability) in enumerate(zip(scores, probabilities)):
    print(f"Class {class_index}: score={score:+.2f}, probability={probability:.4f}")
print("Probability sum:", probabilities.sum())
print("Predicted class:", np.argmax(probabilities))


## 9. Cross-entropy measures classification error

For a one-hot target $\mathbf{y}$ and predicted probabilities
$\hat{\mathbf{y}}$:

$$
L=-\sum_i y_i\log(\hat y_i).
$$

Only the correct class has $y_i=1$, so for one example:

$$
L=-\log(p_{correct}).
$$

A confident correct prediction has a small loss. A confident wrong prediction
has a large loss.


In [ ]:
correct_class_probabilities = np.array([0.99, 0.90, 0.70, 0.50, 0.10, 0.01])
losses = -np.log(correct_class_probabilities)

print("p(correct)   cross-entropy")
for probability, loss in zip(correct_class_probabilities, losses):
    print(f"   {probability:5.2f}          {loss:6.3f}")

plt.figure(figsize=(7, 4))
p = np.linspace(0.01, 1.0, 300)
plt.plot(p, -np.log(p), linewidth=2)
plt.xlabel("Probability assigned to the correct class")
plt.ylabel("Cross-entropy loss")
plt.title("Low confidence in the correct class produces high loss")
plt.grid(alpha=0.25)
plt.show()


## 10. Train, validation, and test sets

- **Training set:** updates weights and biases.
- **Validation set:** selects hyperparameters and the best checkpoint.
- **Test set:** estimates final performance once, after model choices finish.

**Data leakage** occurs when test information influences training or model
selection. A single random split is not cross-validation. In $k$-fold
cross-validation, training repeats $k$ times while each fold serves as the
validation set once.


## 11. Load handwritten-digit images

In Colab, this cell normally loads the `28 x 28` MNIST dataset. The offline
fallback contains `8 x 8` handwritten digits; it enlarges and centers them on a
`28 x 28` canvas. The fallback is useful for testing the notebook, but it does
not replace MNIST for reporting MNIST results.


In [ ]:
def resize_digits_to_mnist(images_8x8):
    # Nearest-neighbor resize from 8 x 8 to 24 x 24, then pad to 28 x 28.
    enlarged = np.repeat(np.repeat(images_8x8, 3, axis=1), 3, axis=2)
    return np.pad(enlarged, ((0, 0), (2, 2), (2, 2)), mode="constant")


try:
    from tensorflow.keras.datasets import mnist

    (X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()
    DATASET_NAME = "MNIST"
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=10_000,
        random_state=SEED,
        stratify=y_train_full,
    )
except Exception as error:
    warnings.warn(f"MNIST unavailable ({type(error).__name__}); using offline digits fallback.")
    digits = load_digits()
    images = resize_digits_to_mnist(digits.images)
    labels = digits.target
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        images, labels, test_size=0.20, random_state=SEED, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.20,
        random_state=SEED,
        stratify=y_train_val,
    )
    DATASET_NAME = "scikit-learn digits resized to 28 x 28 (offline fallback)"

# Keep the from-scratch NumPy training responsive during an in-class Colab run.
if len(X_train) > 20_000:
    all_indices = np.arange(len(X_train))
    selected_indices, _ = train_test_split(
        all_indices,
        train_size=20_000,
        random_state=SEED,
        stratify=y_train,
    )
    X_train = X_train[selected_indices]
    y_train = y_train[selected_indices]

pixel_scale = float(np.max(X_train))
X_train = X_train.astype(np.float32) / pixel_scale
X_val = X_val.astype(np.float32) / pixel_scale
X_test = X_test.astype(np.float32) / pixel_scale

print("Dataset:", DATASET_NAME)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:      ", X_test.shape, y_test.shape)
print("Pixel range:", (X_train.min(), X_train.max()))


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for ax, image, label in zip(axes.ravel(), X_train[:16], y_train[:16]):
    ax.imshow(image, vmin=0, vmax=1)
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.suptitle(f"Training examples: {DATASET_NAME}")
plt.tight_layout()
plt.show()


## 12. One-layer 10-class image classifier

The lecture baseline has no hidden layer:

$$
28\times28\text{ image}\rightarrow784\text{ values}
\rightarrow10\text{ scores}\rightarrow\text{softmax}\rightarrow\text{digit}.
$$

Its parameters are

$$
784\times10+10=7{,}850.
$$

The `ml4a` visualization referenced in the lecture uses ten sigmoid outputs.
This notebook uses **softmax** because MNIST assigns exactly one class to each
image. Sigmoid outputs are independent and need not sum to one; softmax outputs
form one probability distribution across the ten digits. Both approaches can
select the largest output with `argmax`.

For a batch $X$, the forward pass is

$$
Z=XW+b,\qquad P=\operatorname{softmax}(Z).
$$

With softmax cross-entropy, the score gradient is

$$
\frac{\partial L}{\partial Z}=\frac{P-Y}{N}.
$$

### What the optimizer does

The optimizer uses these gradients to change the parameters in a direction
that reduces loss. The learning rate controls the update size. This
from-scratch implementation uses mini-batch stochastic gradient descent. More
advanced optimizers include momentum and Adam.


In [ ]:
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)


def one_hot(labels, class_count=10):
    encoded = np.zeros((len(labels), class_count), dtype=np.float32)
    encoded[np.arange(len(labels)), labels] = 1.0
    return encoded


def predict_probabilities(X, W, b):
    return softmax(X @ W + b)


def evaluate_loss_accuracy(X, y, W, b):
    probabilities = predict_probabilities(X, W, b)
    loss = -np.mean(np.log(probabilities[np.arange(len(y)), y] + 1e-12))
    accuracy = np.mean(np.argmax(probabilities, axis=1) == y)
    return float(loss), float(accuracy)


def train_linear_softmax(
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=20,
    batch_size=256,
    learning_rate=0.25,
    l2=1e-4,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.01, size=(X_train.shape[1], 10))
    b = np.zeros(10)
    y_train_encoded = one_hot(y_train)

    best = {"val_loss": np.inf, "W": W.copy(), "b": b.copy(), "epoch": 0}
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        indices = rng.permutation(len(X_train))
        for start in range(0, len(X_train), batch_size):
            batch_indices = indices[start:start + batch_size]
            xb = X_train[batch_indices]
            yb = y_train_encoded[batch_indices]

            probabilities = predict_probabilities(xb, W, b)
            gradient_scores = (probabilities - yb) / len(xb)
            gradient_W = xb.T @ gradient_scores + l2 * W
            gradient_b = gradient_scores.sum(axis=0)

            W -= learning_rate * gradient_W
            b -= learning_rate * gradient_b

        train_loss, train_acc = evaluate_loss_accuracy(X_train, y_train, W, b)
        val_loss, val_acc = evaluate_loss_accuracy(X_val, y_val, W, b)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_loss < best["val_loss"]:
            best = {"val_loss": val_loss, "W": W.copy(), "b": b.copy(), "epoch": epoch}

        print(
            f"Epoch {epoch:02d}/{epochs}: "
            f"train loss={train_loss:.4f}, val loss={val_loss:.4f}, "
            f"train acc={train_acc:.3f}, val acc={val_acc:.3f}"
        )

    return best["W"], best["b"], history, best["epoch"]


W, b, history, best_epoch = train_linear_softmax(
    X_train_flat,
    y_train,
    X_val_flat,
    y_val,
    epochs=20,
)

print(f"\nBest validation-loss checkpoint: epoch {best_epoch}")
print("Weight shape:", W.shape)
print("Bias shape:  ", b.shape)
print("Parameters:  ", W.size + b.size)


In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="Training")
axes[0].plot(epochs, history["val_loss"], label="Validation")
axes[0].axvline(best_epoch, color="tab:red", linestyle="--", label="Best checkpoint")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(epochs, history["train_acc"], label="Training")
axes[1].plot(epochs, history["val_acc"], label="Validation")
axes[1].axvline(best_epoch, color="tab:red", linestyle="--")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


## 13. Final test evaluation

Use the test set only after choosing the model and checkpoint from validation
performance. The confusion matrix places true classes on rows and predicted
classes on columns. Correct predictions lie on the diagonal.


In [ ]:
test_probabilities = predict_probabilities(X_test_flat, W, b)
test_predictions = np.argmax(test_probabilities, axis=1)
test_accuracy = accuracy_score(y_test, test_predictions)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test,
    test_predictions,
    labels=np.arange(10),
    zero_division=0,
)
matrix = confusion_matrix(y_test, test_predictions, labels=np.arange(10))

print(f"Dataset: {DATASET_NAME}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nclass  precision  recall  F1    support")
for digit in range(10):
    print(
        f"  {digit:>2d}      {precision[digit]:.3f}    {recall[digit]:.3f}  "
        f"{f1[digit]:.3f}    {support[digit]:4d}"
    )

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(matrix, cmap="Blues")
for row in range(10):
    for col in range(10):
        color = "white" if matrix[row, col] > matrix.max() * 0.55 else "black"
        ax.text(col, row, matrix[row, col], ha="center", va="center", color=color, fontsize=8)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("Predicted digit")
ax.set_ylabel("True digit")
ax.set_title("Test confusion matrix")
fig.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## 14. Error analysis

Accuracy says how often the model is wrong. Error analysis helps explain why.
The next cell finds the largest off-diagonal confusion and displays confident
mistakes from that class pair.


In [ ]:
off_diagonal = matrix.copy()
np.fill_diagonal(off_diagonal, 0)
true_class, predicted_class = np.unravel_index(np.argmax(off_diagonal), off_diagonal.shape)

pair_mask = (y_test == true_class) & (test_predictions == predicted_class)
pair_indices = np.where(pair_mask)[0]
pair_indices = pair_indices[np.argsort(test_probabilities[pair_indices, predicted_class])[::-1]]

print(
    f"Largest confusion: true {true_class} predicted as {predicted_class} "
    f"({off_diagonal[true_class, predicted_class]} examples)"
)

if len(pair_indices) == 0:
    print("No off-diagonal mistakes were found.")
else:
    show_count = min(8, len(pair_indices))
    fig, axes = plt.subplots(1, show_count, figsize=(2 * show_count, 2.4))
    axes = np.atleast_1d(axes)
    for ax, index in zip(axes, pair_indices[:show_count]):
        ax.imshow(X_test[index], vmin=0, vmax=1)
        confidence = test_probabilities[index, predicted_class]
        ax.set_title(f"true {y_test[index]}\npred {predicted_class}: {confidence:.2f}")
        ax.axis("off")
    plt.suptitle("High-confidence mistakes from the largest confusion cell")
    plt.tight_layout()
    plt.show()


## 15. What each output neuron learned

Each column of $W$ contains the pixel weights for one class. Reshaping a column
to `28 x 28` shows which pixel locations increase or decrease that class score.
These are linear templates, not complete digit images.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
weight_limit = np.max(np.abs(W))
for digit, ax in enumerate(axes.ravel()):
    view = ax.imshow(
        W[:, digit].reshape(28, 28),
        cmap="coolwarm",
        vmin=-weight_limit,
        vmax=weight_limit,
    )
    ax.set_title(f"Output neuron {digit}")
    ax.axis("off")
fig.colorbar(view, ax=axes.ravel().tolist(), shrink=0.75, label="Learned weight")
plt.suptitle("Learned 28 x 28 weight template for each digit")
plt.show()


## 16. Precision, recall, and threshold choice

For a chosen positive class:

$$
\text{Precision}=\frac{TP}{TP+FP},\qquad
\text{Recall}=\frac{TP}{TP+FN}.
$$

- Prefer high **recall** when missing a positive is costly, such as screening.
- Prefer high **precision** when a false alarm is costly, such as automatic
  account suspension.
- Use precision, recall, F1, and the confusion matrix together when both error
  types matter.

Treat digit `8` as the positive class and vary the probability threshold.


In [ ]:
positive_class = 8
binary_truth = (y_test == positive_class).astype(int)
positive_probability = test_probabilities[:, positive_class]

print("threshold  precision  recall   F1")
for threshold in [0.10, 0.25, 0.50, 0.75, 0.90]:
    binary_prediction = (positive_probability >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        binary_truth,
        binary_prediction,
        average="binary",
        zero_division=0,
    )
    print(f"   {threshold:4.2f}       {p:5.3f}    {r:5.3f}  {f:5.3f}")


## 17. Cross-validation concept

The following cell shows the fold assignments without retraining the neural
network. Each sample appears in a validation fold once and in a training fold
four times.


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
seen_as_validation = np.zeros(len(y_train), dtype=int)

for fold_number, (train_indices, validation_indices) in enumerate(
    folds.split(X_train_flat, y_train), start=1
):
    seen_as_validation[validation_indices] += 1
    print(
        f"Fold {fold_number}: train={len(train_indices):5d}, "
        f"validation={len(validation_indices):5d}"
    )

print("Every sample used for validation exactly once:", np.all(seen_as_validation == 1))


## 18. Why fully connected image models do not scale well

Flattening discards explicit spatial structure. A dense first layer also needs
one separate weight for every input-to-neuron connection.

Assume 128 hidden neurons:

| Input | Flattened values | First-layer weights |
|---|---:|---:|
| `28 x 28` gray | 784 | 100,352 |
| `256 x 256` RGB | 196,608 | 25,165,824 |
| `512 x 512` RGB | 786,432 | 100,663,296 |

Convolutional neural networks address this by preserving spatial arrangement
and sharing kernel weights across image locations. CNNs are the subject of the
next lecture.


## 19. In-class exercises

### Exercise A: neuron calculation

For $\mathbf{x}=[0.3,0.9,0.4]$, $\mathbf{w}=[-0.5,0.7,0.2]$, and $b=-0.1$:

1. Calculate $z=\mathbf{x}^T\mathbf{w}+b$.
2. Calculate $\sigma(z)$.
3. Apply a threshold of `0.5`.

### Exercise B: parameters

Count all trainable parameters in `784 -> 64 -> 32 -> 10`.

### Exercise C: softmax and loss

For scores $[2.0,1.0,0.0]$ and correct class `1`:

1. Calculate the softmax probabilities.
2. Calculate the cross-entropy loss.

### Exercise D: evaluation

A detector has `TP=72`, `FP=18`, and `FN=8`. Calculate precision, recall, and
F1. Which error would you prioritize for a cancer-screening application?

### Exercise E: experiments

1. Change the softmax-classifier learning rate to `0.02`, `0.10`, and `0.50`.
2. Compare convergence and validation performance.
3. Change the random seed and record the result.
4. Inspect the largest confusion pair and propose one data or model change.


In [ ]:
# Run this cell after solving the exercises by hand.

# Exercise A
exercise_x = np.array([0.3, 0.9, 0.4])
exercise_w = np.array([-0.5, 0.7, 0.2])
exercise_b = -0.1
exercise_z = exercise_x @ exercise_w + exercise_b
exercise_a = sigmoid(exercise_z)

# Exercise B
_, exercise_parameter_total = dense_parameter_count([784, 64, 32, 10])

# Exercise C
exercise_probabilities = softmax(np.array([2.0, 1.0, 0.0]))
exercise_loss = -np.log(exercise_probabilities[1])

# Exercise D
tp, fp, fn = 72, 18, 8
exercise_precision = tp / (tp + fp)
exercise_recall = tp / (tp + fn)
exercise_f1 = 2 * exercise_precision * exercise_recall / (exercise_precision + exercise_recall)

print("Exercise A")
print(f"z={exercise_z:.4f}, sigmoid(z)={exercise_a:.4f}, class={int(exercise_a >= 0.5)}")
print("\nExercise B")
print(f"Total parameters: {exercise_parameter_total:,}")
print("\nExercise C")
print("Probabilities:", np.round(exercise_probabilities, 4))
print(f"Cross-entropy for correct class 1: {exercise_loss:.4f}")
print("\nExercise D")
print(
    f"Precision={exercise_precision:.4f}, recall={exercise_recall:.4f}, "
    f"F1={exercise_f1:.4f}"
)
print("Cancer screening usually prioritizes recall to reduce false negatives.")


## 20. Interactive parameter-count problems

Use the controls to explore how the number of neurons and layers changes the number of trainable parameters. Include one bias for every output neuron.

For a dense layer: parameters = inputs × neurons + neurons for biases.


In [ ]:
def parameter_count_from_layers(layer_sizes):
    rows = []
    total = 0
    for layer_number, (n_in, n_out) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]), start=1):
        weights = n_in * n_out
        biases = n_out
        layer_total = weights + biases
        rows.append((layer_number, n_in, n_out, weights, biases, layer_total))
        total += layer_total
    return rows, total

def show_parameter_count(layer_sizes):
    rows, total = parameter_count_from_layers(layer_sizes)
    print(f"Architecture: {' -> '.join(map(str, layer_sizes))}")
    print("Layer | Inputs | Neurons | Weights | Biases | Total")
    print("-" * 58)
    for layer, n_in, n_out, weights, biases, layer_total in rows:
        print(f"{layer:>5} | {n_in:>6} | {n_out:>7} | {weights:>7,} | {biases:>6,} | {layer_total:>7,}")
    print(f"Total trainable parameters: {total:,}")
    return total

show_parameter_count([784, 64, 32, 10])

In [ ]:
# Interactive version for Google Colab
import ipywidgets as widgets
from IPython.display import display

input_size = widgets.IntSlider(value=784, min=2, max=1024, description='Input')
hidden_1 = widgets.IntSlider(value=64, min=1, max=256, description='Hidden 1')
hidden_2 = widgets.IntSlider(value=32, min=0, max=256, description='Hidden 2')
output_size = widgets.IntSlider(value=10, min=2, max=50, description='Output')

def interactive_count(input_size, hidden_1, hidden_2, output_size):
    layers = [input_size, hidden_1] + ([hidden_2] if hidden_2 > 0 else []) + [output_size]
    show_parameter_count(layers)

ui = widgets.interactive_output(interactive_count, {"input_size": input_size, "hidden_1": hidden_1, "hidden_2": hidden_2, "output_size": output_size})
display(input_size, hidden_1, hidden_2, output_size, ui)

### Challenge questions

1. Which layer contributes the most parameters in `784 -> 64 -> 32 -> 10`?
2. What happens when the first hidden layer changes from 64 to 128 neurons?
3. Compare `784 -> 128 -> 10` with `784 -> 64 -> 64 -> 10`. Which has more parameters?
4. How many parameters are in a dense layer from a `32 x 32 x 3` colour image to 100 neurons?
5. Why can adding neurons increase capacity and also increase overfitting risk?

In [ ]:
for name, architecture in {
    'A': [784, 64, 32, 10],
    'B': [784, 128, 10],
    'C': [784, 64, 64, 10],
    'Colour image to dense layer': [32 * 32 * 3, 100],
}.items():
    print(f'\n{name}')
    show_parameter_count(architecture)

## 21. Homework from LN7

1. Replicate the AND, XOR, and handwritten-digit experiments.
2. Find the smallest neural network that solves AND reliably.
3. Search for a small hidden-layer size that improves digit accuracy over the
   one-layer baseline.
4. Report training, validation, and test performance separately.
5. Include a confusion matrix and representative mistakes.

## Summary

- A neuron computes a weighted sum, adds a bias, and applies an activation.
- Nonlinear activations let multilayer networks learn nonlinear decisions.
- One perceptron can solve AND but cannot solve XOR.
- Softmax converts class scores to probabilities that sum to one.
- Cross-entropy strongly penalizes low probability on the correct class.
- Validation data guides model selection; test data measures final performance.
- Accuracy alone can hide class-specific errors. Use precision, recall, F1,
  confusion matrices, and image-level error analysis.
- Flattened dense models provide a useful baseline but scale poorly for large
  images, motivating convolutional neural networks.
